In [1]:
import os
import csv
import warnings
import numpy as np
import xarray as xr
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.isotonic import IsotonicRegression
#from extreme_verif_plots import load_ingredients, filter_categories

In [16]:
EPS = 1e-12

LABEL_ORDER = {"cont": ["hits", "miss", "fa", "cn"],
               "eno": ["event", "nonevent"],
               "sse3": ["sse_fcst", "sse_ref", "n"]}

In [17]:
def _arr(ds, key, dims):
    """Read ds[key] with label-enforced order on cont/eno/sse3, transposed."""
    da = ds[key]
    for dim, labels in LABEL_ORDER.items():
        if dim in da.dims:
            da = da.sel({dim: labels})
    if "pbin" in da.dims:
        da = da.sortby("pbin")
    return da.transpose(*dims).values

def _units(var, accum):
    if "precip" in var.lower():
        return "mm hr$^{-1}$" if accum == "hourly" else "mm day$^{-1}$"
    if "spd" in var.lower() or "wind" in var.lower():
        return "m s$^{-1}$"
    return ""

def _year_from_path(p, i):
    import os, re
    m = re.search(r'(19|20)\d{2}', os.path.basename(p))
    return m.group(0) if m else f'src{i}'
    
def scores_from_contingency(c):
    """(hits, miss, fa, cn) -> dict of categorical scores incl. SEDI."""
    hits, miss, fa, cn = (float(x) for x in c)
    n = hits + miss + fa + cn
    pod = hits / (hits + miss + EPS)            # H = hit rate
    far = fa / (hits + fa + EPS)
    sr = 1.0 - far                               # success ratio
    csi = hits / (hits + miss + fa + EPS)        # threat score
    fbias = (hits + fa) / (hits + miss + EPS)
    pofd = fa / (fa + cn + EPS)                   # F = false-alarm rate
    # SEDI (Ferro & Stephenson 2011); base-rate independent, non-degenerate.
    H = np.clip(pod, EPS, 1 - EPS)
    F = np.clip(pofd, EPS, 1 - EPS)
    sedi = ((np.log(F) - np.log(H) - np.log(1 - F) + np.log(1 - H))
            / (np.log(F) + np.log(H) + np.log(1 - F) + np.log(1 - H)))
    return dict(POD=pod, SR=sr, FAR=far, CSI=csi, FBIAS=fbias,
                POFD=pofd, SEDI=sedi, base_rate=(hits + miss) / (n + EPS), n=n)

def fss_from_components(comp):
    """comp: (..., 2) additive sums [sum (Pf-Po)^2, sum (Pf^2+Po^2)] -> FSS."""
    comp = np.asarray(comp, float)
    num = comp[..., 0]; den = comp[..., 1]
    return np.where(den > 1e-15, 1.0 - num / np.maximum(den, 1e-15), np.nan)

def auc_from_hist(H):
    """
    Exact AUC (Mann-Whitney, ties=0.5) from a [nbins, 2] event/non-event table
    whose rows are in increasing score order. Works for the discrete ensemble
    table and for fine continuous histograms.
    """
    n1 = H[:, 0].astype(float)            # events per bin (increasing score)
    n0 = H[:, 1].astype(float)            # non-events per bin
    N1, N0 = n1.sum(), n0.sum()
    if N1 == 0 or N0 == 0:
        return np.nan
    cum_n0_below = np.concatenate([[0.0], np.cumsum(n0)[:-1]])   # non-events at lower score
    # for each event in bin k: full credit for n0 below, half credit for ties in k
    num = np.sum(n1 * (cum_n0_below + 0.5 * n0))
    return float(num / (N1 * N0))

def optimal_threshold_scores(H):
    """
    Categorical scores of the ensemble used as a binary forecast at EVERY
    probability cutoff k/M (exact from the histogram). Returns per-cutoff CSI
    and SEDI arrays plus the max and its p*. Selection-based -- label as
    'optimal probability threshold' when reporting.
    """
    M = H.shape[0] - 1
    n1 = H[:, 0].astype(float)
    n0 = H[:, 1].astype(float)
    N1, N0 = n1.sum(), n0.sum()
    csi = np.full(M + 1, np.nan)
    sedi = np.full(M + 1, np.nan)
    for k in range(M + 1):                       # predict-yes when prob >= k/M
        hits = n1[k:].sum(); fa = n0[k:].sum()
        miss = N1 - hits; cn = N0 - fa
        s = scores_from_contingency(np.array([hits, miss, fa, cn]))
        csi[k], sedi[k] = s['CSI'], s['SEDI']
    p = np.arange(M + 1) / M
    kc = int(np.nanargmax(csi)); ks = int(np.nanargmax(sedi))
    return dict(p=p, CSI=csi, SEDI=sedi,
                CSI_opt=float(csi[kc]), p_CSI=float(p[kc]),
                SEDI_opt=float(sedi[ks]), p_SEDI=float(p[ks]))

def filter_categories(ing, categories):
    """Return a storm-filtered copy of one variable's ingredient dict."""
    keep = np.isin(ing["categories"], list(categories))
    if keep.sum() == 0:
        raise ValueError(f"category filter {sorted(categories)} kept no storms")
    new = dict(ing, storms=ing["storms"][keep], categories=ing["categories"][keep],
               unet_tables=ing["unet_tables"][:, keep],
               unet_score_hist=ing["unet_score_hist"][:, keep],
               target_hist=(ing["target_hist"][keep]
                            if ing.get("target_hist") is not None else None),
               unet_fss=(ing["unet_fss"][:, keep]
                         if ing.get("unet_fss") is not None else None),
               ens={e: {k: (a[:, keep] if a is not None else None)
                        for k, a in d.items()}
                    for e, d in ing["ens"].items()})
    return new

def load_ingredients(sources, ensembles=("ERA5", "GDAS"), labels=None):
    """
    Open/accept one or more per-year ingredient sources -- Zarr paths and/or
    already-opened xr.Datasets -- and concatenate the per-storm ingredients
    along the storm axis (storms relabeled '{label}_{name}' when multiple
    sources are given; labels default to the year parsed from the filename, or
    'src{i}' for datasets). Thresholds and score edges must match across
    sources. Returns {var: dict(...)}.
    """
    opened = _normalize_sources(sources, labels)
    multi = len(opened) > 1
    varnames = sorted({k[:-len("_score_edges")] for _, d in opened
                       for k in d.data_vars if k.endswith("_score_edges")})
    out = {}
    for v in varnames:
        tdim, sdim, bdim = f"{v}_thr", f"{v}_storm", f"{v}_sbin"
        present = [(yr, d) for yr, d in opened if f"{v}_unet_tables" in d]
        thr_raw = present[0][1][tdim].values
        # old ingredient files carry float thresholds; new ones carry labels
        thr_labels = np.array([str(t) for t in np.atleast_1d(thr_raw)])
        if f"{v}_thr_value" in present[0][1]:
            thr_values = present[0][1][f"{v}_thr_value"].values.astype(float)
        else:
            try:
                thr_values = np.asarray(thr_raw, float)
            except (TypeError, ValueError):
                thr_values = np.full(thr_labels.size, np.nan)
        edges = present[0][1][f"{v}_score_edges"].values
        accum = present[0][1].attrs.get(f"{v}_accum", "hourly")
        for yr, d in present[1:]:
            if not np.array_equal(np.array([str(t) for t in d[tdim].values]),
                                  thr_labels):
                raise ValueError(f"{v}: thresholds differ across files")
            if not np.allclose(d[f"{v}_score_edges"].values, edges):
                raise ValueError(f"{v}: score edges differ across files")
        storms = np.concatenate(
            [[f"{yr}_{s}" if multi else str(s) for s in d[sdim].values]
             for yr, d in present])
        catkey = f"{v}_storm_category"
        cats = np.concatenate(
            [d[catkey].values if catkey in d
             else np.full(d.sizes[sdim], -1, np.int64) for _, d in present])

        def cat_(key, dims):
            return np.concatenate([_arr(d, key, dims) for _, d in present], axis=1)

        unit = _units(v, accum)
        thr_disp = [f"{val:g} {unit}" if np.isfinite(val)
                    else f"{lab} (per-cell)"
                    for lab, val in zip(thr_labels, thr_values)]
        ing = dict(
            thr=thr_labels, thr_values=thr_values, thr_disp=thr_disp,
            edges=np.asarray(edges, float),
            accum=accum, storms=storms, categories=cats, M=None,
            unet_tables=cat_(f"{v}_unet_tables", (tdim, sdim, "cont")),
            unet_score_hist=cat_(f"{v}_unet_score_hist",
                                 (tdim, sdim, bdim, "eno")),
            target_hist=(np.concatenate(
                             [_arr(d, f"{v}_target_hist", (sdim, bdim))
                              for _, d in present], axis=0)
                         if f"{v}_target_hist" in present[0][1] else None),
            fss_scales=(present[0][1][f"{v}_fss_scales"].values
                        if f"{v}_fss_scales" in present[0][1] else None),
            unet_fss=(cat_(f"{v}_unet_fss", (tdim, sdim, f"{v}_scale",
                                             "fsscomp"))
                      if f"{v}_unet_fss" in present[0][1] else None),
            ens={})
        for ens in ensembles:
            if f"{v}_{ens}_prob_hist" not in present[0][1]:
                continue
            ing["ens"][ens] = dict(
                member_tables=cat_(f"{v}_{ens}_member_tables",
                                   (tdim, sdim, "member", "cont")),
                prob_hist=cat_(f"{v}_{ens}_prob_hist",
                               (tdim, sdim, "pbin", "eno")),
                cellclimo_sse=cat_(f"{v}_{ens}_cellclimo_sse",
                                   (tdim, sdim, "sse3")),
                member_score_hist=cat_(f"{v}_{ens}_member_score_hist",
                                       (tdim, sdim, "member", bdim, "eno")),
                member_fss=(cat_(f"{v}_{ens}_member_fss",
                                 (tdim, sdim, "member", f"{v}_scale", "fsscomp"))
                            if f"{v}_{ens}_member_fss" in present[0][1] else None),
                prob_fss=(cat_(f"{v}_{ens}_prob_fss",
                               (tdim, sdim, f"{v}_scale", "fsscomp"))
                          if f"{v}_{ens}_prob_fss" in present[0][1] else None))
            ing["M"] = ing["ens"][ens]["member_tables"].shape[2]
        out[v] = ing
    return out

def _normalize_sources(sources, labels=None):
    """Accept path(s) and/or opened xr.Dataset(s); return [(label, Dataset)]."""
    if isinstance(sources, (str, xr.Dataset)):
        sources = [sources]
    sources = list(sources)
    if labels is not None and len(labels) != len(sources):
        raise ValueError("labels must match the number of sources")
    opened = []
    for i, s in enumerate(sources):
        if isinstance(s, str):
            lab = labels[i] if labels is not None else _year_from_path(s, i)
            opened.append((str(lab), xr.open_zarr(s)))
        elif isinstance(s, xr.Dataset):
            lab = labels[i] if labels is not None else f"src{i}"
            opened.append((str(lab), s))
        else:
            raise TypeError(f"source {i}: expected path str or xr.Dataset, "
                            f"got {type(s)}")
    return opened

In [1]:
"""
classification_metrics.py -- combined 2020-2024 classification scorecard.

Reads the per-year ingredient Zarrs produced by extreme_verif_tc.py, pools all
selected TC cases EXACTLY (per-storm additive ingredients summed across storms
and years -- never averaging finished scores), and tabulates classification
metrics for each (variable, threshold):

    AUC, FSS (one column per neighborhood scale), TS (= CSI, threat score),
    ETS (Gilbert skill score), POD, FAR

Rows per (variable, threshold):
    UNet            binary forecast at the threshold; AUC is the AUC of the
                    continuous UNet field
    {ENS}_members   mean over the M members, each member verified as its OWN
                    binary forecast (never the ensemble-mean field)
    {ENS}_prob      ensemble exceedance probability: AUC exact (Mann-Whitney)
                    from the (M+1)-bin histogram; FSS of the probability
                    field; TS/ETS/POD/FAR at the CSI-optimal probability
                    cutoff p* -- a SELECTION, reported in the p_opt column

FSS columns appear only when the ingredient Zarrs carry neighborhood
components (extreme_verif_tc run with fss_scales); on older files they are
'--'. Scales are neighborhood widths in grid points (x8 km on the 8-km grid).

Requires extreme_verif_tc.py and extreme_verif_plots.py on the Python path.

Usage:
    python classification_metrics.py                      # config in __main__
or:
    from classification_metrics import classification_table, write_csv
    from extreme_verif_plots import load_ingredients
    ing = load_ingredients(paths)                         # or opened Datasets
    rows = classification_table(ing['WRF_precip'], 'WRF_precip',
                                thresholds=['p90', 'p95'])
"""

# ----------------------------------------------------------------------
# helpers
# ----------------------------------------------------------------------
def _table_at_cutoff(H, k):
    """Contingency table of 'predict yes when prob >= k/M' from the hist."""
    n1 = H[:, 0].astype(float)
    n0 = H[:, 1].astype(float)
    hits, fa = n1[k:].sum(), n0[k:].sum()
    return np.array([hits, n1.sum() - hits, fa, n0.sum() - fa])


def select_threshold_indices(ing, wanted):
    """
    Resolve a user threshold list to indices. Entries may be labels
    (e.g. 'p90', '20') or numeric values matched against the stored threshold
    values (e.g. 20, 40/24). None -> all thresholds in the file.
    """
    labels = [str(t) for t in ing["thr"]]
    if wanted is None:
        return list(range(len(labels)))
    vals = np.asarray(ing["thr_values"], float)
    idxs = []
    for w in wanted:
        if isinstance(w, str) and w in labels:
            idxs.append(labels.index(w))
            continue
        try:
            wv = float(w)
        except (TypeError, ValueError):
            raise KeyError(f"threshold {w!r} not found; available labels: "
                           f"{labels}")
        j = (int(np.nanargmin(np.abs(vals - wv)))
             if np.isfinite(vals).any() else -1)
        if j >= 0 and np.isfinite(vals[j]) and abs(vals[j] - wv) < 1e-6:
            idxs.append(j)
        else:
            raise KeyError(f"threshold {w!r} not found; available labels "
                           f"{labels}, values {np.round(vals, 5)}")
    return idxs


# ----------------------------------------------------------------------
# core table
# ----------------------------------------------------------------------
def classification_table(ing, var, thresholds=None):
    """
    Build the classification-metric rows for one variable from pooled
    ingredients (all storms of all loaded sources summed exactly).
    Returns a list of row dicts.
    """
    scales = (np.asarray(ing["fss_scales"], int)
              if ing.get("fss_scales") is not None else None)

    def fss_cols(fss_vals):
        if scales is None:
            return {}
        if fss_vals is None:
            return {f"FSS_s{s}": np.nan for s in scales}
        return {f"FSS_s{s}": float(fss_vals[i]) for i, s in enumerate(scales)}

    rows = []
    for ti in select_threshold_indices(ing, thresholds):
        base = dict(var=var, threshold=str(ing["thr"][ti]),
                    display=ing["thr_disp"][ti])

        # ---- UNet: deterministic binary + continuous-field AUC ----
        s = scores_from_contingency(ing["unet_tables"][ti].sum(0))
        u_fss = (fss_from_components(ing["unet_fss"][ti].sum(0))
                 if ing.get("unet_fss") is not None else None)
        rows.append(dict(base, source="UNet", p_opt=np.nan,
                         AUC=auc_from_hist(ing["unet_score_hist"][ti].sum(0)),
                         **fss_cols(u_fss),
                         TS=s["CSI"], ETS=s["ETS"], POD=s["POD"], FAR=s["FAR"]))

        for ens, d in ing["ens"].items():
            # ---- members: each member as its own binary forecast ----
            mt = d["member_tables"][ti]                       # (storm, M, 4)
            M = mt.shape[1]
            per = [scores_from_contingency(mt[:, m].sum(0)) for m in range(M)]
            msh = d["member_score_hist"][ti]
            auc_m = [auc_from_hist(msh[:, m].sum(0)) for m in range(M)]
            m_fss = (np.nanmean(fss_from_components(
                         d["member_fss"][ti].sum(0)), axis=0)
                     if d.get("member_fss") is not None else None)
            rows.append(dict(
                base, source=f"{ens}_members", p_opt=np.nan,
                AUC=float(np.nanmean(auc_m)), **fss_cols(m_fss),
                TS=float(np.mean([p["CSI"] for p in per])),
                ETS=float(np.nanmean([p["ETS"] for p in per])),
                POD=float(np.mean([p["POD"] for p in per])),
                FAR=float(np.mean([p["FAR"] for p in per]))))

            # ---- ensemble probability: exact AUC + optimal-p* categorical ----
            H = d["prob_hist"][ti].sum(0)
            opt = optimal_threshold_scores(H)
            k = int(np.nanargmax(opt["CSI"])) if np.isfinite(opt["CSI"]).any() \
                else 0
            sp = scores_from_contingency(_table_at_cutoff(H, k))
            p_fss = (fss_from_components(d["prob_fss"][ti].sum(0))
                     if d.get("prob_fss") is not None else None)
            rows.append(dict(
                base, source=f"{ens}_prob", p_opt=float(opt["p"][k]),
                AUC=auc_from_hist(H), **fss_cols(p_fss),
                TS=sp["CSI"], ETS=sp["ETS"], POD=sp["POD"], FAR=sp["FAR"]))
    return rows


# ----------------------------------------------------------------------
# output
# ----------------------------------------------------------------------
def _fss_keys(rows):
    return sorted({k for r in rows for k in r if k.startswith("FSS_s")},
                  key=lambda k: int(k[5:]))


def print_table(rows):
    if not rows:
        return
    cols = ["AUC"] + _fss_keys(rows) + ["TS", "ETS", "POD", "FAR", "p_opt"]
    seen = set()
    for r in rows:
        key = (r["var"], r["threshold"])
        if key not in seen:
            seen.add(key)
            print(f"\n== {r['var']} >= {r['display']} ==")
            print("  " + f"{'source':<15s}"
                  + "".join(f"{c:>9s}" for c in cols))
        line = "  " + f"{r['source']:<15s}"
        for c in cols:
            v = r.get(c, np.nan)
            line += f"{v:>9.3f}" if np.isfinite(v) else f"{'--':>9s}"
        print(line)


def write_csv(rows, path):
    cols = (["var", "threshold", "display", "source", "p_opt", "AUC"]
            + _fss_keys(rows) + ["TS", "ETS", "POD", "FAR"])
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow(r)
    print(f"\nclassification metrics -> {path}")


def make_classification_metrics(sources, outpath, thresholds=None,
                                categories=None, labels=None):
    """
    Driver: pool the given ingredient sources (paths and/or opened Datasets),
    build the table for every variable, print it, and write the CSV.
    thresholds: None (all) or {var: [labels/values]}; categories: optional
    Saffir-Simpson filter, e.g. {3, 4, 5}.
    """
    all_ing = load_ingredients(sources, labels=labels)
    rows = []
    for var, ing in all_ing.items():
        if categories is not None:
            ing = filter_categories(ing, categories)
        print(f"[{var}] {ing['storms'].size} TC cases combined; "
              f"thresholds {list(map(str, ing['thr']))}"
              + ("" if ing.get("fss_scales") is not None else
                 "  (no FSS ingredients in these files -- re-run "
                 "extreme_verif_tc with fss_scales to populate FSS)"))
        sel = thresholds.get(var) if isinstance(thresholds, dict) else thresholds
        rows += classification_table(ing, var, thresholds=sel)
    print_table(rows)
    write_csv(rows, outpath)
    return rows


# ----------------------------------------------------------------------
if __name__ == "__main__":
    base = "/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC"
    SCOREDIR = f"{base}/verif_scores"
    ACCUM = "hourly"
    YEARS = [2020, 2021, 2022, 2023, 2024]

    # None -> every threshold stored in the Zarrs. To restrict, list labels
    # (as shown by the thr coordinate) and/or numeric values, e.g.:
    # THRESHOLDS = {'WRF_precip': [40/24, 80/24, 'p90', 'p95'],
    #               'WRF_SPD10':  [20, 30, 'p90', 'p95']}
    THRESHOLDS = None
    CATEGORIES = None                       # e.g. {3, 4, 5} for majors only

    paths = []
    for y in YEARS:
        p = f"{SCOREDIR}/extreme_scores_{y}_{ACCUM}.zarr"
        if os.path.exists(p):
            paths.append(p)
        else:
            warnings.warn(f"missing {p} -- skipping")
    if not paths:
        raise SystemExit("no ingredient Zarrs found")
    print(f"pooling {len(paths)} year file(s)")

    tag = "" if CATEGORIES is None else "_cat" + "".join(map(str,
                                                             sorted(CATEGORIES)))
    make_classification_metrics(
        paths, f"{SCOREDIR}/classification_metrics_{ACCUM}{tag}.csv",
        thresholds=THRESHOLDS, categories=CATEGORIES)